In [4]:
import jax
print(jax.devices())
!pip install flax optax -q

import jax
import jax.numpy as jnp
import optax
import time
from flax import linen as nn

# Minimal MLP
class MLP(nn.Module):
    hidden: int
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(self.hidden)(x)
        x = nn.relu(x)
        x = nn.Dense(self.hidden)(x)
        return x

# Setup
model = MLP(hidden=2048)
optimizer = optax.adam(1e-3)
key = jax.random.PRNGKey(0)

x = jax.random.normal(key, (256, 1024))
y = jax.random.normal(key, (256, 2048))

params = model.init(key, x)
opt_state = optimizer.init(params)

@jax.jit
def train_step(params, opt_state, x, y):
    def loss_fn(params):
        pred = model.apply(params, x)
        return jnp.mean((pred - y) ** 2)
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss

# Benchmark
p, o = params, opt_state
for _ in range(5):  # warmup
    p, o, l = train_step(p, o, x, y)
l.block_until_ready()

times = []
for _ in range(50):
    t0 = time.time()
    p, o, l = train_step(p, o, x, y)
    l.block_until_ready()
    times.append(time.time() - t0)

print(f"Device: {jax.devices()[0]}")
print(f"Min step time:  {min(times):.4f}s")
print(f"Avg step time:  {sum(times)/len(times):.4f}s")
print(f"Final loss:     {float(l):.6f}")
print(f"Throughput:     {256/( sum(times)/len(times)):.1f} samples/sec")

[CudaDevice(id=0), CudaDevice(id=1)]
Device: cuda:0
Min step time:  0.0028s
Avg step time:  0.0031s
Final loss:     0.005057
Throughput:     83411.4 samples/sec
